In [1]:
import uuid
from functools import reduce
from pyspark import StorageLevel
from pyspark.sql import functions as F
from pyspark.sql.window import Window

RUN_ID = str(uuid.uuid4())[:8]
META_COLS = ["_source_file", "_ingest_time", "_run_id"]

# Pattern of an ST37 well identifier, for example 00/06-06-001-01W4/0
UWI_PATTERN = r"^(\w{2})/(\d{2})-(\d{2})-(\d{3})-(\d{2})W(\d)/(\d+)$"


def normalize_strings(sdf, exclude=()):
    # Trim every string column and convert empty strings to NULL
    for c, t in sdf.dtypes:
        if t == "string" and c not in exclude:
            trimmed = F.trim(F.col(c))
            sdf = sdf.withColumn(c, F.when(trimmed == "", F.lit(None)).otherwise(trimmed))
    return sdf


def is_masked(raw_col):
    # Petrinex masks confidential values with ***
    return F.coalesce(F.col(raw_col) == "***", F.lit(False))


def unexpected_failure(raw_col, typed_col):
    return F.col(raw_col).isNotNull() & F.col(typed_col).isNull() & ~is_masked(raw_col)


def add_well_key(sdf, uwi_col="Well_UWI"):
    # Build the Petrinex style well key from an ST37 UWI
    u = F.upper(F.col(uwi_col))
    p = [F.regexp_extract(u, UWI_PATTERN, i) for i in range(1, 8)]
    key = F.concat(F.lit("1"), p[0], p[1], p[2], p[3], p[4], F.lit("W"), p[5], F.lpad(p[6], 2, "0"))
    return sdf.withColumn("well_key", F.when(u.rlike(UWI_PATTERN), key))


def check_columns(table, required):
    cols = spark.table(table).columns
    missing = [c for c in required if c not in cols]
    if missing:
        raise ValueError(f"{table} is missing {missing}. Run the upstream cell first.")


print("RUN_ID:", RUN_ID)


StatementMeta(, 9e7e0bb9-b8a4-46a7-8702-e3fa98508c87, 3, Finished, Available, Finished, False)

RUN_ID: ac23b6a9


In [2]:
# Silver well dimension: union of ST37 production_string and BH, one row per well_key
ps = add_well_key(normalize_strings(spark.table("bronze_st37_production_string"), exclude=META_COLS))
bh = add_well_key(normalize_strings(spark.table("bronze_st37_bh"), exclude=META_COLS))

print("production_string rows:", ps.count(), " null well_key:", ps.filter(F.col("well_key").isNull()).count())
print("bh rows:", bh.count(), " null well_key:", bh.filter(F.col("well_key").isNull()).count())

ps_sel = (ps.filter(F.col("well_key").isNotNull())
    .select("well_key",
            F.col("Well_UWI").alias("well_uwi"),
            F.col("Well_Licence_Number").alias("licence_number"),
            F.col("Well_Name").alias("well_name"),
            F.col("Status_Fluid").alias("status_fluid"),
            F.col("Status_Mode").alias("status_mode"),
            F.col("Status_Type").alias("status_type"))
    .withColumn("in_production_string", F.lit(True)))

bh_sel = (bh.filter(F.col("well_key").isNotNull())
    .select("well_key",
            F.col("Well_UWI").alias("bh_uwi"),
            F.col("Well_Licence_Number").alias("bh_licence_number"),
            F.col("Well_Name").alias("bh_well_name"),
            F.col("Licence_Status").alias("licence_status"))
    .withColumn("in_bh", F.lit(True)))

dim_new = (ps_sel.join(bh_sel, "well_key", "full_outer")
    .select("well_key",
            F.coalesce("well_uwi", "bh_uwi").alias("well_uwi"),
            F.coalesce("licence_number", "bh_licence_number").alias("licence_number"),
            F.coalesce("well_name", "bh_well_name").alias("well_name"),
            "status_fluid", "status_mode", "status_type", "licence_status",
            F.coalesce("in_production_string", F.lit(False)).alias("in_production_string"),
            F.coalesce("in_bh", F.lit(False)).alias("in_bh"),
            F.lit(False).alias("is_placeholder"),
            F.lit(RUN_ID).alias("_silver_run_id")))

dim_new.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_dim_well")

d = spark.table("silver_dim_well")
print("dim rows:", d.count(), " distinct well_key:", d.select("well_key").distinct().count())
display(d.groupBy("in_production_string", "in_bh").count())


StatementMeta(, 9e7e0bb9-b8a4-46a7-8702-e3fa98508c87, 4, Finished, Available, Finished, False)

production_string rows: 658266  null well_key: 0
bh rows: 538629  null well_key: 0
dim rows: 664984  distinct well_key: 664984


SynapseWidget(Synapse.DataFrame, 662208c1-7e16-4772-90fe-acd9ce59e49b)

In [3]:
# Silver Petrinex volumes: clean, cast, flag masked values, then reload by source file
bronze = spark.table("bronze_petrinex_vol")
s = normalize_strings(bronze, exclude=META_COLS)

# Keep the raw text before casting
s = (s
    .withColumn("volume_raw", F.col("Volume"))
    .withColumn("energy_raw", F.col("Energy"))
    .withColumn("hours_raw", F.col("Hours")))

s = (s
    .withColumn("production_month", F.to_date(F.concat(F.col("ProductionMonth"), F.lit("-01"))))
    .withColumn("submission_date", F.to_date(F.col("SubmissionDate")))
    .withColumn("volume", F.expr("try_cast(volume_raw as decimal(18,3))"))
    .withColumn("energy", F.expr("try_cast(energy_raw as decimal(18,3))"))
    .withColumn("hours", F.expr("try_cast(hours_raw as int)"))
    .withColumn("well_key", F.when(F.col("FromToIDType") == "WI", F.upper(F.col("FromToIDIdentifier"))))
    .withColumn("volume_masked", is_masked("volume_raw"))
    .withColumn("energy_masked", is_masked("energy_raw"))
    .withColumn("hours_masked", is_masked("hours_raw"))
    .withColumn("cast_failed_unexpected",
                unexpected_failure("volume_raw", "volume")
                | unexpected_failure("energy_raw", "energy")
                | unexpected_failure("hours_raw", "hours"))
    .withColumn("_silver_run_id", F.lit(RUN_ID)))

files = sorted(r[0] for r in bronze.select("_source_file").distinct().collect())
print("files to load:", len(files), files)

if spark.catalog.tableExists("silver_petrinex_vol"):
    for f in files:
        spark.sql(f"DELETE FROM silver_petrinex_vol WHERE _source_file = '{f}'")

s.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("silver_petrinex_vol")

check_columns("silver_petrinex_vol",
              ["volume_raw", "energy_raw", "hours_raw", "volume_masked", "energy_masked",
               "hours_masked", "cast_failed_unexpected", "well_key", "production_month"])

silver = spark.table("silver_petrinex_vol")
print("silver rows:", silver.count(), " bronze rows:", bronze.count())
print("masked volume:", silver.filter("volume_masked").count(),
      " masked hours:", silver.filter("hours_masked").count(),
      " unexpected failures:", silver.filter("cast_failed_unexpected").count())
display(silver.groupBy("production_month").count().orderBy("production_month"))


StatementMeta(, 9e7e0bb9-b8a4-46a7-8702-e3fa98508c87, 5, Finished, Available, Finished, False)

files to load: 12 ['Vol_2025-07-AB.CSV', 'Vol_2025-08-AB.CSV', 'Vol_2025-09-AB.CSV', 'Vol_2025-10-AB.CSV', 'Vol_2025-11-AB.CSV', 'Vol_2025-12-AB.CSV', 'Vol_2026-01-AB.CSV', 'Vol_2026-02-AB.CSV', 'Vol_2026-03-AB.CSV', 'Vol_2026-04-AB.CSV', 'Vol_2026-05-AB.CSV', 'Vol_2026-06-AB.CSV']
silver rows: 6481961  bronze rows: 6481961
masked volume: 73  masked hours: 7094  unexpected failures: 0


SynapseWidget(Synapse.DataFrame, 6ac546b7-5d78-43a2-ad41-ffc5e89fb709)

In [4]:
# Data quality rules: counts for every rule in one pass, rejected rows in one table
sp = spark.table("silver_petrinex_vol")
dim = spark.table("silver_dim_well")

check_columns("silver_petrinex_vol", ["volume_masked", "hours_masked", "cast_failed_unexpected"])

key_cols = ["production_month", "ReportingFacilityID", "ActivityID", "ProductID", "FromToID"]
business_cols = [c for c in sp.columns if c not in META_COLS + ["_silver_run_id"]]
max_hours = F.dayofmonth(F.last_day("production_month")) * 24

base = (sp
    .withColumn("_rn", F.row_number().over(Window.partitionBy(*business_cols).orderBy(F.lit(1))))
    .withColumn("_key_count", F.count("*").over(Window.partitionBy(*key_cols)))
    .join(dim.select("well_key", F.lit(True).alias("_in_dim")), "well_key", "left"))
base = base.persist(StorageLevel.MEMORY_AND_DISK)

is_prod = F.col("ActivityID") == "PROD"
rules = [
    ("exact_duplicate", "reject", F.col("_rn") > 1),
    ("prod_negative_volume", "reject", is_prod & (F.col("volume") < 0)),
    ("prod_missing_volume", "reject", is_prod & F.col("volume").isNull()),
    ("hours_above_month_max", "reject", F.col("hours") > max_hours),
    ("prod_well_not_in_dim", "warning",
        is_prod & (F.col("FromToIDType") == "WI") & F.col("_in_dim").isNull()),
    ("duplicate_key_masked_fromto", "warning",
        (F.col("FromToID") == "***") & (F.col("_key_count") > 1)),
    ("masked_volume", "info", F.coalesce(F.col("volume_masked"), F.lit(False))),
    ("masked_hours", "info", F.coalesce(F.col("hours_masked"), F.lit(False))),
    ("unexpected_cast_failure", "warning", F.col("cast_failed_unexpected")),
]

out_cols = key_cols + ["volume", "_source_file"]

# One pass over the data for the counts of all rules
count_exprs = [F.sum(F.when(F.coalesce(cond, F.lit(False)), 1).otherwise(0)).alias(name)
               for name, _, cond in rules]
counts = base.agg(*count_exprs).collect()[0].asDict()
dq_rows = [(RUN_ID, "silver", "silver_petrinex_vol", name, sev, int(counts[name] or 0))
           for name, sev, _ in rules]

# Rejected rows: one row per reject rule that a row violates
reject_tags = F.array(*[F.when(F.coalesce(cond, F.lit(False)), F.lit(name))
                        for name, sev, cond in rules if sev == "reject"])
rejected = (base
    .withColumn("rule_name", F.explode(reject_tags))
    .filter(F.col("rule_name").isNotNull())
    .select(*out_cols, "rule_name")
    .withColumn("run_id", F.lit(RUN_ID)))

spark.createDataFrame(dq_rows, "run_id string, layer string, table_name string, rule_name string, severity string, rows_flagged long") \
    .write.format("delta").mode("append").saveAsTable("dq_results")
rejected.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("rejected_records")

print(len(rules), "rules")
display(spark.table("dq_results").filter(F.col("run_id") == RUN_ID).orderBy("rule_name"))
print("rejected rows:", spark.table("rejected_records").filter(F.col("run_id") == RUN_ID).count())


StatementMeta(, 9e7e0bb9-b8a4-46a7-8702-e3fa98508c87, 6, Finished, Available, Finished, False)

9 rules


SynapseWidget(Synapse.DataFrame, 84a272dd-1086-4e17-9be8-c2c37cfcb616)

rejected rows: 567


In [5]:
# Gold tables and reconciliation tests
reject_cond = reduce(lambda a, b: a | b,
    [F.coalesce(cond, F.lit(False)) for _, sev, cond in rules if sev == "reject"])
in_scope = F.coalesce(is_prod & (F.col("FromToIDType") == "WI"), F.lit(False))

flagged = (base
    .withColumn("_rejected", reject_cond)
    .withColumn("_in_scope", in_scope))

fact_rows = flagged.filter(~F.col("_rejected") & F.col("_in_scope"))

# 1) Gold well dimension: silver dimension plus placeholder rows for wells missing from ST37
missing = (fact_rows.select("well_key").distinct()
    .join(dim.select("well_key"), "well_key", "left_anti"))

placeholders = missing
for c, t in dim.dtypes:
    if c != "well_key":
        placeholders = placeholders.withColumn(c, F.lit(None).cast(t))
placeholders = (placeholders
    .withColumn("in_production_string", F.lit(False))
    .withColumn("in_bh", F.lit(False))
    .withColumn("is_placeholder", F.lit(True))
    .select(dim.columns))

gold_dim = dim.unionByName(placeholders)
gold_dim.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_well")
print("placeholder wells:", placeholders.count())

# 2) Gold fact table
fact = (fact_rows
    .select("production_month", "well_key", "ProductID", "ReportingFacilityID",
            "OperatorBAID", "OperatorName", "volume", "energy", "hours",
            "submission_date", "_source_file")
    .withColumn("_gold_run_id", F.lit(RUN_ID)))
fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_fact_production")

# 3) Reconciliation tests
g = spark.table("gold_fact_production")
gd = spark.table("gold_dim_well")
n_silver = sp.count()
n_rejected = flagged.filter(F.col("_rejected")).count()
n_out_of_scope = flagged.filter(~F.col("_rejected") & ~F.col("_in_scope")).count()

grain = ["production_month", "ReportingFacilityID", "well_key", "ProductID"]
dup_grain = g.groupBy(*grain).count().filter("count > 1").count()
orphans = g.join(gd.select("well_key"), "well_key", "left_anti").count()

silver_sum = fact_rows.groupBy("ProductID").agg(F.sum("volume").alias("s"))
gold_sum = g.groupBy("ProductID").agg(F.sum("volume").alias("g"))
sum_mismatch = (silver_sum.join(gold_sum, "ProductID", "full_outer")
    .filter(F.abs(F.coalesce("s", F.lit(0)) - F.coalesce("g", F.lit(0))) > 0.001)
    .count())

tests = [
    ("bronze_equals_silver_rows", spark.table("bronze_petrinex_vol").count(), n_silver),
    ("silver_equals_rejected_plus_out_of_scope_plus_gold", n_silver, n_rejected + n_out_of_scope + g.count()),
    ("gold_volume_totals_match_silver_by_product", 0, sum_mismatch),
    ("gold_fact_grain_unique", 0, dup_grain),
    ("gold_fact_no_orphan_wells", 0, orphans),
]
rows = [(RUN_ID, n, int(e), int(a), "PASS" if e == a else "FAIL") for n, e, a in tests]
spark.createDataFrame(rows, "run_id string, test_name string, expected long, actual long, status string") \
    .write.format("delta").mode("append").saveAsTable("reconciliation_results")

display(spark.table("reconciliation_results").filter(F.col("run_id") == RUN_ID))
print("gold fact rows:", g.count(), " gold dim rows:", gd.count())


StatementMeta(, 9e7e0bb9-b8a4-46a7-8702-e3fa98508c87, 7, Finished, Available, Finished, False)

placeholder wells: 9


SynapseWidget(Synapse.DataFrame, 6c9d8b7f-3a14-48b2-bfb2-8c1b4eb79eb5)

gold fact rows: 2908544  gold dim rows: 664993


In [6]:
# Test the reject rules with constructed bad rows. Nothing is written to Silver, Gold or rejected_records.

# One real production row is used as a template for every case
seed_rows = (sp.filter(is_prod & (F.col("FromToIDType") == "WI")
                       & F.col("volume").isNotNull() & F.col("hours").isNotNull())
               .join(dim.select("well_key"), "well_key", "left_semi")
               .select(sp.columns)
               .limit(1).collect())
seed = spark.createDataFrame(seed_rows, sp.schema)


def make_case(name, **changes):
    d = seed
    for col, val in changes.items():
        d = d.withColumn(col, F.lit(val).cast(dict(seed.dtypes)[col]))
    return d.withColumn("test_case", F.lit(name))


cases = [
    make_case("clean_row", volume=1.234),
    make_case("negative_prod", volume=-100.0),
    make_case("missing_prod_volume", volume=None),
    make_case("hours_too_high", hours=99999),
    make_case("exact_duplicate"),
    make_case("exact_duplicate"),
    make_case("unknown_well", well_key="199999999999W400", volume=2.345),
]
test_df = reduce(lambda a, b: a.unionByName(b), cases)

# Same helper columns and same rules list as the real pipeline
tb = (test_df
    .withColumn("_rn", F.row_number().over(Window.partitionBy(*business_cols).orderBy(F.lit(1))))
    .withColumn("_key_count", F.count("*").over(Window.partitionBy(*key_cols)))
    .join(dim.select("well_key", F.lit(True).alias("_in_dim")), "well_key", "left"))

for name, sev, cond in rules:
    tb = tb.withColumn(f"hit_{name}", F.coalesce(cond, F.lit(False)))

agg = tb.groupBy("test_case").agg(*[F.sum(F.col(f"hit_{n}").cast("int")).alias(n) for n, _, _ in rules])
actual = {r["test_case"]: {n: r[n] for n, _, _ in rules if r[n]} for r in agg.collect()}

expected = {
    "clean_row": {},
    "negative_prod": {"prod_negative_volume": 1},
    "missing_prod_volume": {"prod_missing_volume": 1},
    "hours_too_high": {"hours_above_month_max": 1},
    "exact_duplicate": {"exact_duplicate": 1},     # two identical rows: only the second one is rejected
    "unknown_well": {"prod_well_not_in_dim": 1},   # warning only, the row is kept
}

results = []
for case_name, exp in expected.items():
    act = actual.get(case_name, {})
    results.append((case_name, str(exp), str(act), "PASS" if exp == act else "FAIL"))

# Rows that would reach Gold: clean_row, unknown_well and one copy of the duplicate
reject_any = reduce(lambda a, b: a | b, [F.col(f"hit_{n}") for n, sev, _ in rules if sev == "reject"])
kept = tb.filter(~reject_any).count()
results.append(("rows_kept_for_gold", "3", str(kept), "PASS" if kept == 3 else "FAIL"))

res_df = spark.createDataFrame(results, "test_case string, expected string, actual string, status string")
res_df.write.format("delta").mode("overwrite").saveAsTable("dq_rule_test_results")
display(res_df)


StatementMeta(, 9e7e0bb9-b8a4-46a7-8702-e3fa98508c87, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8f735c81-57e6-4484-a1f4-8ab0a611a4db)

In [7]:
# Gap detection: missing calendar months and wells that appear or disappear between months
from dateutil.relativedelta import relativedelta

months_df = sp.groupBy("production_month").count().orderBy("production_month")
display(months_df)
loaded = [r["production_month"] for r in months_df.collect()]

missing_months = []
if loaded:
    cur = loaded[0]
    while cur < loaded[-1]:
        cur = cur + relativedelta(months=1)
        if cur not in loaded:
            missing_months.append(cur)
print("loaded months:", len(loaded), " missing calendar months:", missing_months)

wells = (sp.filter((F.col("ActivityID") == "PROD") & (F.col("FromToIDType") == "WI"))
           .select("production_month", "well_key").distinct())
wells = wells.persist(StorageLevel.MEMORY_AND_DISK)

gap_rows = [(RUN_ID, "silver", "silver_petrinex_vol", "missing_calendar_month", "warning", len(missing_months))]

coverage_rows = [(RUN_ID, loaded[0],
                  wells.filter(F.col("production_month") == F.lit(loaded[0])).count(),
                  None, None)]

for prev, curr in zip(loaded[:-1], loaded[1:]):
    a = wells.filter(F.col("production_month") == F.lit(prev)).select("well_key")
    b = wells.filter(F.col("production_month") == F.lit(curr)).select("well_key")
    n_prev, n_curr = a.count(), b.count()
    gone = a.join(b, "well_key", "left_anti").count()
    new = b.join(a, "well_key", "left_anti").count()
    coverage_rows.append((RUN_ID, curr, n_curr, new, gone))
    print(prev, "->", curr, "| wells:", n_prev, n_curr, "| disappeared:", gone, "| new:", new)
    gap_rows.append((RUN_ID, "silver", "silver_petrinex_vol", f"wells_disappeared_{prev}_to_{curr}", "warning", gone))
    gap_rows.append((RUN_ID, "silver", "silver_petrinex_vol", f"wells_new_{prev}_to_{curr}", "info", new))

spark.createDataFrame(gap_rows, "run_id string, layer string, table_name string, rule_name string, severity string, rows_flagged long") \
    .write.format("delta").mode("append").saveAsTable("dq_results")

spark.createDataFrame(
    coverage_rows,
    "run_id string, production_month date, producing_wells long, wells_added long, wells_dropped long"
).write.format("delta").mode("append").saveAsTable("dq_month_coverage")


StatementMeta(, 9e7e0bb9-b8a4-46a7-8702-e3fa98508c87, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 47ff1440-34b4-4aad-8f5b-f6b0477fd56a)

loaded months: 12  missing calendar months: []
2025-07-01 -> 2025-08-01 | wells: 135870 135088 | disappeared: 2220 | new: 1438
2025-08-01 -> 2025-09-01 | wells: 135088 135393 | disappeared: 1605 | new: 1910
2025-09-01 -> 2025-10-01 | wells: 135393 135284 | disappeared: 2155 | new: 2046
2025-10-01 -> 2025-11-01 | wells: 135284 135962 | disappeared: 1633 | new: 2311
2025-11-01 -> 2025-12-01 | wells: 135962 135387 | disappeared: 1884 | new: 1309
2025-12-01 -> 2026-01-01 | wells: 135387 135445 | disappeared: 1580 | new: 1638
2026-01-01 -> 2026-02-01 | wells: 135445 135413 | disappeared: 1427 | new: 1395
2026-02-01 -> 2026-03-01 | wells: 135413 135660 | disappeared: 1526 | new: 1773
2026-03-01 -> 2026-04-01 | wells: 135660 135401 | disappeared: 1775 | new: 1516
2026-04-01 -> 2026-05-01 | wells: 135401 135154 | disappeared: 2077 | new: 1830
2026-05-01 -> 2026-06-01 | wells: 135154 135154 | disappeared: 1751 | new: 1751


In [8]:
# Summary of this run
dq_this_run = spark.table("dq_results").filter(F.col("run_id") == RUN_ID)
print("dq_results rows for this run:", dq_this_run.count())
display(dq_this_run.orderBy("rule_name"))

display(spark.table("reconciliation_results").filter(F.col("run_id") == RUN_ID))
display(spark.table("dq_rule_test_results"))

print("gold fact rows:", spark.table("gold_fact_production").count(),
      " gold dim rows:", spark.table("gold_dim_well").count())

from datetime import datetime

spark.createDataFrame([(RUN_ID, datetime.utcnow())], "run_id string, run_ts timestamp") \
    .write.format("delta").mode("append").saveAsTable("dq_run_log")
base.unpersist()
wells.unpersist()


StatementMeta(, 9e7e0bb9-b8a4-46a7-8702-e3fa98508c87, 10, Finished, Available, Finished, False)

dq_results rows for this run: 32


SynapseWidget(Synapse.DataFrame, 10dd6b8a-ffe6-4f1a-b1c3-129d0f164546)

SynapseWidget(Synapse.DataFrame, 79c4e005-1980-4020-882d-d19dc206c000)

SynapseWidget(Synapse.DataFrame, 56b7c328-df82-4bfd-a378-6b0373dbc8b7)

gold fact rows: 2908544  gold dim rows: 664993


DataFrame[production_month: date, well_key: string]

In [9]:
from pyspark.sql import functions as F

sp = spark.table("silver_petrinex_vol")
dim = spark.table("silver_dim_well")

prod_wells = (sp.filter((F.col("ActivityID") == "PROD") & (F.col("FromToIDType") == "WI"))
                .select("well_key").distinct())
n_wells = prod_wells.count()

n_union = prod_wells.join(dim.select("well_key"), "well_key", "inner").count()
n_ps = prod_wells.join(dim.filter("in_production_string").select("well_key"), "well_key", "inner").count()

print("distinct producing wells (12 months):", n_wells)
print("found in production_string or BH:", n_union, " rate:", round(n_union / n_wells, 4))
print("found in production_string alone:", n_ps, " rate:", round(n_ps / n_wells, 4))

StatementMeta(, 9e7e0bb9-b8a4-46a7-8702-e3fa98508c87, 11, Finished, Available, Finished, False)

distinct producing wells (12 months): 144259
found in production_string or BH: 144250  rate: 0.9999
found in production_string alone: 143813  rate: 0.9969


In [10]:
fact = spark.table("gold_fact_production")
gd = spark.table("gold_dim_well").select("well_key", "in_production_string")

j = fact.join(gd, "well_key", "left")
missing_cond = ~F.coalesce(F.col("in_production_string"), F.lit(False))

share = (j.groupBy("ProductID")
    .agg(F.sum("volume").alias("total_volume"),
         F.sum(F.when(missing_cond, F.col("volume")).otherwise(0)).alias("volume_from_missing_wells"))
    .withColumn("share", F.round(F.col("volume_from_missing_wells") / F.col("total_volume"), 4)))
display(share.filter(F.col("ProductID").isin("GAS", "OIL", "WATER", "COND")))

n_missing = (gd.filter(~F.col("in_production_string"))
               .join(fact.select("well_key").distinct(), "well_key").count())
print("wells in Gold fact that are missing from production_string:", n_missing)

StatementMeta(, 9e7e0bb9-b8a4-46a7-8702-e3fa98508c87, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0a6c5a12-d7da-4ab0-9d70-09ac23252999)

wells in Gold fact that are missing from production_string: 446
